# 目标检测基础

本notebook介绍目标检测的核心概念和基础技术。

## 学习目标

- 理解边界框的概念和表示方法
- 掌握锚框的生成机制
- 了解多尺度检测原理
- 实践:目标检测数据集的使用

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
from PIL import Image

print(f"PyTorch版本: {torch.__version__}")

## 1. 边界框 (Bounding Box)

### 1.1 什么是边界框?

**定义**: 用矩形框标注图像中对象的位置

**目标检测 vs 图像分类**:
```
图像分类:
  输入: 图像
  输出: 类别 (如"猫")

目标检测:
  输入: 图像
  输出: 多个 (类别, 边界框)
         如 [("狗", [60,45,378,516]),
             ("猫", [400,112,655,493])]
```

### 1.2 两种表示方法

**方法1: 左上角-右下角 (Corner Format)**
```
(x1, y1, x2, y2)
  ↓
(x1,y1)━━━━━━┓
┃          ┃
┃  Object  ┃
┃          ┃
┗━━━━━━(x2,y2)
```
- `x1, y1`: 左上角坐标
- `x2, y2`: 右下角坐标

**方法2: 中心-宽高 (Center Format)**
```
(cx, cy, w, h)
  ↓
┏━━━━━━━━━━┓
┃    ●(cx,cy)
┃  Object  ┃ h
┃          ┃
┗━━━━━━━━━━┛
     w
```
- `cx, cy`: 中心点坐标
- `w, h`: 宽度和高度

### 1.3 坐标系统

```
图像坐标系:
(0,0)───→ x
 │
 │
 ↓
 y

注意:
  - 原点在左上角
  - x轴向右
  - y轴向下
```

In [ ]:
# 边界框转换函数
def box_corner_to_center(boxes):
    """从(左上,右下)转换到(中心,宽高)
    
    Args:
        boxes: shape (n, 4), 每行是 [x1, y1, x2, y2]
    Returns:
        shape (n, 4), 每行是 [cx, cy, w, h]
    """
    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    w = x2 - x1
    h = y2 - y1
    boxes = torch.stack((cx, cy, w, h), axis=-1)
    return boxes

def box_center_to_corner(boxes):
    """从(中心,宽高)转换到(左上,右下)
    
    Args:
        boxes: shape (n, 4), 每行是 [cx, cy, w, h]
    Returns:
        shape (n, 4), 每行是 [x1, y1, x2, y2]
    """
    cx, cy, w, h = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    x1 = cx - 0.5 * w
    y1 = cy - 0.5 * h
    x2 = cx + 0.5 * w
    y2 = cy + 0.5 * h
    boxes = torch.stack((x1, y1, x2, y2), axis=-1)
    return boxes

# 测试转换
print("=== 边界框格式转换 ===")
dog_bbox = torch.tensor([[60.0, 45.0, 378.0, 516.0]])
print(f"Corner格式: {dog_bbox}")

center = box_corner_to_center(dog_bbox)
print(f"Center格式: {center}")

corner = box_center_to_corner(center)
print(f"转回Corner: {corner}")

print("\n解释:")
print(f"  中心点: ({center[0,0]:.1f}, {center[0,1]:.1f})")
print(f"  宽度: {center[0,2]:.1f}")
print(f"  高度: {center[0,3]:.1f}")

### 1.4 可视化边界框

In [ ]:
def bbox_to_rect(bbox, color):
    """将边界框转换为matplotlib格式"""
    return Rectangle(
        xy=(bbox[0], bbox[1]),
        width=bbox[2]-bbox[0],
        height=bbox[3]-bbox[1],
        fill=False,
        edgecolor=color,
        linewidth=2
    )

# 创建示例图像(如果没有实际图像)
print("创建示例图像...")
img_array = np.random.randint(0, 255, (600, 700, 3), dtype=np.uint8)

# 绘制边界框
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(img_array)

# 添加边界框
dog_bbox_list = [60, 45, 378, 516]
cat_bbox_list = [400, 112, 655, 493]

ax.add_patch(bbox_to_rect(dog_bbox_list, 'blue'))
ax.add_patch(bbox_to_rect(cat_bbox_list, 'red'))

# 添加标签
ax.text(dog_bbox_list[0], dog_bbox_list[1]-5, 'Dog', 
        fontsize=12, color='blue', weight='bold')
ax.text(cat_bbox_list[0], cat_bbox_list[1]-5, 'Cat',
        fontsize=12, color='red', weight='bold')

ax.axis('off')
plt.title('边界框示例', fontsize=14)
plt.show()

print("\n边界框信息:")
print(f"  狗: {dog_bbox_list}")
print(f"  猫: {cat_bbox_list}")

## 2. 锚框 (Anchor Box)

### 2.1 为什么需要锚框?

**问题**: 目标可能出现在图像的任何位置,大小和形状各异

**解决方案**: 锚框
- 在图像上预先生成大量候选框
- 不同尺度和宽高比
- 作为检测的起始点

**类比**: 钓鱼时在池塘多个位置放鱼钩
```
没有锚框:           有锚框:
  在哪找目标? →     预先放置候选框
  搜索空间大 →      减少搜索范围
```

### 2.2 锚框生成原理

**参数**:
- **尺度 (Scale)** $s \in (0, 1]$: 控制大小
- **宽高比 (Aspect Ratio)** $r > 0$: 控制形状

**尺寸计算**:
$$
\text{width} = hs\sqrt{r}
$$
$$
\text{height} = \frac{hs}{\sqrt{r}}
$$

其中 $h$ 是图像高度。

**示例**:
- $s=0.75, r=1$: 正方形,边长 $0.75h$
- $s=0.75, r=2$: 宽矩形,宽 $0.75h\sqrt{2}$,高 $0.75h/\sqrt{2}$
- $s=0.75, r=0.5$: 高矩形,宽 $0.75h/\sqrt{2}$,高 $0.75h\sqrt{2}$

### 2.3 多尺度锚框策略

**朴素方法**: 所有组合
- 尺度: $s_1, s_2, ..., s_n$
- 宽高比: $r_1, r_2, ..., r_m$
- 锚框数: $n \times m$ (太多!)

**高效方法**: 特定组合
- $(s_1, r_1), (s_1, r_2), ..., (s_1, r_m)$ ← 固定尺度,变化宽高比
- $(s_2, r_1), (s_3, r_1), ..., (s_n, r_1)$ ← 固定宽高比,变化尺度
- 总数: $n + m - 1$ (大幅减少!)

**示例**:
```
尺度: [0.75, 0.5, 0.25]
宽高比: [1, 2, 0.5]

生成锚框:
(0.75,1), (0.75,2), (0.75,0.5) ← 3个
(0.5,1), (0.25,1)              ← 2个
━━━━━━━━━━━━━━━━━━━━━━━━━━━━
总共: 3+3-1 = 5个/像素
```

**全图锚框数**:
- 图像大小: $w \times h$
- 每像素锚框: $a = n+m-1$
- 总锚框数: $w \times h \times a$
- 示例: $561 \times 728 \times 5 = 2,042,040$

In [ ]:
# 锚框生成函数
def multibox_prior(data, sizes, ratios):
    """生成以每个像素为中心的锚框
    
    Args:
        data: 输入特征图, shape (batch, channels, h, w)
        sizes: 尺度列表, 如 [0.75, 0.5, 0.25]
        ratios: 宽高比列表, 如 [1, 2, 0.5]
    
    Returns:
        锚框, shape (1, num_anchors, 4)
        坐标格式: (cx, cy, w, h), 归一化到 [0, 1]
    """
    in_height, in_width = data.shape[-2:]
    device = data.device
    num_sizes, num_ratios = len(sizes), len(ratios)
    boxes_per_pixel = (num_sizes + num_ratios - 1)
    
    size_tensor = torch.tensor(sizes, device=device)
    ratio_tensor = torch.tensor(ratios, device=device)
    
    # 为了将锚点移动到像素中心,设置偏移量
    offset_h, offset_w = 0.5, 0.5
    steps_h = 1.0 / in_height
    steps_w = 1.0 / in_width
    
    # 生成锚框的所有中心点
    center_h = (torch.arange(in_height, device=device) + offset_h) * steps_h
    center_w = (torch.arange(in_width, device=device) + offset_w) * steps_w
    shift_y, shift_x = torch.meshgrid(center_h, center_w, indexing='ij')
    shift_y, shift_x = shift_y.reshape(-1), shift_x.reshape(-1)
    
    # 生成boxes_per_pixel个高和宽
    # 第一个尺度配所有宽高比
    w = torch.cat((size_tensor * torch.sqrt(ratio_tensor[0]),
                   sizes[0] * torch.sqrt(ratio_tensor[1:])))
    h = torch.cat((size_tensor / torch.sqrt(ratio_tensor[0]),
                   sizes[0] / torch.sqrt(ratio_tensor[1:])))
    
    # 除以2来获得半高和半宽
    anchor_manipulations = torch.stack((-w, -h, w, h)).T.repeat(
                                        in_height * in_width, 1) / 2
    
    # 每个中心点都有boxes_per_pixel个锚框
    out_grid = torch.stack([shift_x, shift_y, shift_x, shift_y],
                          dim=1).repeat_interleave(boxes_per_pixel, dim=0)
    output = out_grid + anchor_manipulations
    return output.unsqueeze(0)

# 测试锚框生成
print("=== 锚框生成示例 ===")
img = torch.zeros((1, 3, 561, 728))
sizes = [0.75, 0.5, 0.25]
ratios = [1, 2, 0.5]

anchors = multibox_prior(img, sizes=sizes, ratios=ratios)
print(f"锚框shape: {anchors.shape}")
print(f"  - batch: {anchors.shape[0]}")
print(f"  - 总锚框数: {anchors.shape[1]:,}")
print(f"  - 坐标维度: {anchors.shape[2]}")

boxes_per_pixel = len(sizes) + len(ratios) - 1
print(f"\n每像素锚框数: {boxes_per_pixel}")
print(f"计算: {len(sizes)} + {len(ratios)} - 1 = {boxes_per_pixel}")
print(f"\n锚框组合:")
for i, s in enumerate(sizes):
    if i == 0:
        for r in ratios:
            print(f"  (s={s}, r={r})")
    else:
        print(f"  (s={s}, r={ratios[0]})")

### 2.4 可视化锚框

In [ ]:
def show_bboxes(axes, bboxes, labels=None, colors=None):
    """在图上显示所有边界框"""
    def make_list(obj, default_values=None):
        if obj is None:
            obj = default_values
        elif not isinstance(obj, (list, tuple)):
            obj = [obj]
        return obj
    
    labels = make_list(labels)
    colors = make_list(colors, ['b', 'g', 'r', 'm', 'c'])
    
    for i, bbox in enumerate(bboxes):
        color = colors[i % len(colors)]
        rect = bbox_to_rect(bbox.detach().numpy(), color)
        axes.add_patch(rect)
        if labels and len(labels) > i:
            text_color = 'k' if color == 'w' else 'w'
            axes.text(rect.xy[0], rect.xy[1], labels[i],
                     va='center', ha='center', fontsize=9, color=text_color,
                     bbox=dict(facecolor=color, lw=0))

# 可视化特定位置的锚框
h, w = img.shape[-2:]
print(f"图像大小: {h} x {w}")

# 访问(250, 250)位置的锚框
boxes = anchors.reshape((h, w, boxes_per_pixel, 4))
target_boxes = boxes[250, 250, :, :]

print(f"\n位置(250, 250)的{boxes_per_pixel}个锚框:")
for i in range(boxes_per_pixel):
    print(f"  锚框{i}: {target_boxes[i]}")

# 绘制锚框
fig, ax = plt.subplots(figsize=(8, 8))
img_display = np.random.randint(0, 255, (h, w, 3), dtype=np.uint8)
ax.imshow(img_display)

# 将锚框从归一化坐标转换为像素坐标
bbox_scale = torch.tensor([w, h, w, h])
pixel_boxes = box_center_to_corner(target_boxes) * bbox_scale

show_bboxes(ax, pixel_boxes,
           labels=[f's={sizes[i//len(ratios)]},r={ratios[i%len(ratios)]}' 
                  if i < len(sizes)*len(ratios) else 
                  f's={sizes[i-len(ratios)+1]},r={ratios[0]}'
                  for i in range(boxes_per_pixel)])

ax.set_xlim(0, w)
ax.set_ylim(h, 0)
ax.axis('off')
plt.title(f'像素(250, 250)的{boxes_per_pixel}个锚框', fontsize=14)
plt.show()

## 3. 多尺度目标检测

### 3.1 为什么需要多尺度?

**问题**: 目标大小差异巨大
```
远处的车:  10×10像素
近处的车: 200×200像素
```

**单一尺度的问题**:
- 小锚框: 检测不到大目标
- 大锚框: 检测不到小目标
- 太多锚框: 计算量大

### 3.2 多尺度检测策略

**核心思想**: 在不同尺度的特征图上生成锚框

```
输入图像 (561×728)
    ↓
┌─────────────────────┐
│ 大特征图 (4×4)      │ → 小锚框 (s=0.15)
│ 检测小目标          │    16个位置
└─────────────────────┘
    ↓ 下采样
┌─────────────────────┐
│ 中特征图 (2×2)      │ → 中锚框 (s=0.4)
│ 检测中目标          │    4个位置
└─────────────────────┘
    ↓ 下采样
┌─────────────────────┐
│ 小特征图 (1×1)      │ → 大锚框 (s=0.8)
│ 检测大目标          │    1个位置
└─────────────────────┘
```

**优势**:
1. **减少锚框数**: 从 $561×728×5=2,042,040$ 减少到 $16×5+4×5+1×5=105$
2. **匹配目标尺度**: 小特征图→大感受野→大目标
3. **高效计算**: 特征共享,避免重复

### 3.3 感受野的作用

**感受野**: 特征图上一个单元对应输入图像的区域

```
浅层 (大特征图):
  感受野小 → 细节信息 → 检测小目标

深层 (小特征图):
  感受野大 → 全局信息 → 检测大目标
```

**CNN特性**:
- 逐层下采样: 特征图变小
- 感受野扩大: 看到更大区域
- 语义层次: 底层→边缘, 高层→对象

In [ ]:
# 多尺度锚框生成演示
def display_anchors_multiscale(fmap_w, fmap_h, s):
    """在不同尺度下生成并显示锚框"""
    fmap = torch.zeros((1, 10, fmap_h, fmap_w))
    anchors = multibox_prior(fmap, sizes=s, ratios=[1, 2, 0.5])
    
    print(f"\n特征图: {fmap_h}×{fmap_w}, 尺度: {s}")
    print(f"锚框总数: {anchors.shape[1]}")
    print(f"每像素: {anchors.shape[1] // (fmap_h * fmap_w)}个")
    
    # 可视化
    fig, ax = plt.subplots(figsize=(6, 6))
    # 使用实际图像大小
    img_h, img_w = 561, 728
    img_display = np.random.randint(0, 255, (img_h, img_w, 3), dtype=np.uint8)
    ax.imshow(img_display)
    
    bbox_scale = torch.tensor([img_w, img_h, img_w, img_h])
    pixel_boxes = box_center_to_corner(anchors[0]) * bbox_scale
    show_bboxes(ax, pixel_boxes[:min(20, len(pixel_boxes))])  # 只显示前20个
    
    ax.set_xlim(0, img_w)
    ax.set_ylim(img_h, 0)
    ax.axis('off')
    plt.title(f'特征图{fmap_h}×{fmap_w}, 尺度{s[0]}', fontsize=12)
    plt.show()

print("=== 多尺度锚框生成 ===")
# 尺度1: 小锚框检测小目标
display_anchors_multiscale(4, 4, [0.15])

# 尺度2: 中锚框检测中目标  
display_anchors_multiscale(2, 2, [0.4])

# 尺度3: 大锚框检测大目标
display_anchors_multiscale(1, 1, [0.8])

## 4. 目标检测数据集

### 4.1 香蕉检测数据集

**特点**:
- 1000张合成图像
- 每张图1个香蕉
- 不同位置、角度、大小
- CSV标注文件

**数据格式**:
```
CSV文件:
  image_name, class, xmin, ymin, xmax, ymax
  0.png, 0, 53, 87, 198, 245
  1.png, 0, 120, 45, 231, 198
  ...
```

### 4.2 批处理挑战

**问题**: 不同图像的目标数量不同
```
图像1: 1个目标 → label shape (1, 5)
图像2: 3个目标 → label shape (3, 5)
图像3: 2个目标 → label shape (2, 5)

如何批处理? shape不一致!
```

**解决方案**: 填充到统一大小
```
批大小=3, 最大目标数=3

图像1: [class, x1, y1, x2, y2] + 2个填充
图像2: [c, x, y, x, y] × 3  (无填充)
图像3: [c, x, y, x, y] × 2 + 1个填充

填充值: class = -1 (非法类别)
批shape: (3, 3, 5)
```

In [ ]:
# 模拟香蕉数据集
class BananasDataset(Dataset):
    """香蕉检测数据集"""
    def __init__(self, is_train=True, num_samples=100):
        self.is_train = is_train
        # 生成模拟数据
        self.features = []
        self.labels = []
        
        for i in range(num_samples):
            # 模拟图像 (256×256)
            img = torch.rand(3, 256, 256)
            # 模拟边界框 (归一化坐标)
            x1 = torch.rand(1) * 0.5
            y1 = torch.rand(1) * 0.5
            x2 = x1 + torch.rand(1) * 0.3 + 0.1
            y2 = y1 + torch.rand(1) * 0.3 + 0.1
            # 标签: [class, x1, y1, x2, y2]
            label = torch.tensor([0, x1, y1, x2, y2]).squeeze()
            
            self.features.append(img)
            self.labels.append(label)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

def banana_collate_fn(batch):
    """自定义批处理函数,处理不同数量的目标"""
    images = []
    labels = []
    
    for img, label in batch:
        images.append(img)
        labels.append(label)
    
    images = torch.stack(images)
    # 对于香蕉数据集,每张图只有1个目标
    # 统一shape为 (batch_size, max_boxes, 5)
    labels = torch.stack(labels).unsqueeze(1)  # (batch, 1, 5)
    
    return images, labels

# 创建数据集和加载器
print("=== 香蕉检测数据集 ===")
train_dataset = BananasDataset(is_train=True, num_samples=100)
train_loader = DataLoader(train_dataset, batch_size=4, 
                         shuffle=True, collate_fn=banana_collate_fn)

print(f"训练集大小: {len(train_dataset)}")
print(f"批大小: 4")

# 获取一个批次
images, labels = next(iter(train_loader))
print(f"\n批次shape:")
print(f"  图像: {images.shape}  # (batch, channels, height, width)")
print(f"  标签: {labels.shape}  # (batch, max_boxes, 5)")

print(f"\n第一张图的标签:")
print(f"  {labels[0]}")
print(f"  解释: [类别, x1, y1, x2, y2]")
print(f"  类别0表示香蕉")

# 可视化批次
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    img = images[i].permute(1, 2, 0).numpy()
    axes[i].imshow(img)
    
    # 绘制边界框
    label = labels[i, 0]  # 第一个目标
    if label[0] != -1:  # 不是填充
        # 从归一化坐标转换为像素坐标
        bbox = label[1:] * 256
        rect = bbox_to_rect(bbox.numpy(), 'red')
        axes[i].add_patch(rect)
    
    axes[i].axis('off')
    axes[i].set_title(f'Image {i}')

plt.suptitle('香蕉检测数据集批次示例', fontsize=14)
plt.tight_layout()
plt.show()

## 5. 小结

### 边界框

**两种表示**:
1. Corner格式: `(x1, y1, x2, y2)`
2. Center格式: `(cx, cy, w, h)`

**转换公式**:
$$
\begin{aligned}
\text{Corner→Center:} \quad & c_x = \frac{x_1+x_2}{2}, \quad c_y = \frac{y_1+y_2}{2} \\
& w = x_2-x_1, \quad h = y_2-y_1 \\
\text{Center→Corner:} \quad & x_1 = c_x - \frac{w}{2}, \quad y_1 = c_y - \frac{h}{2} \\
& x_2 = c_x + \frac{w}{2}, \quad y_2 = c_y + \frac{h}{2}
\end{aligned}
$$

### 锚框生成

**参数**:
- 尺度 $s_1, ..., s_n$
- 宽高比 $r_1, ..., r_m$

**尺寸公式**:
$$
w = hs\sqrt{r}, \quad h = \frac{hs}{\sqrt{r}}
$$

**高效策略**:
- 每像素: $n+m-1$ 个锚框
- 全图: $w \times h \times (n+m-1)$ 个

### 多尺度检测

**核心思想**:
```
大特征图 + 小锚框 → 小目标
小特征图 + 大锚框 → 大目标
```

**优势**:
1. 减少锚框数量
2. 匹配目标尺度
3. 利用特征层次

**感受野**:
- 浅层: 小感受野 → 局部细节
- 深层: 大感受野 → 全局语义

### 数据集处理

**挑战**: 目标数量不一致

**解决**: 填充 + 标记
- 填充到 `max_boxes`
- 非法框: `class = -1`
- 批shape: `(batch, max_boxes, 5)`

### 关键要点

1. **边界框**: 目标检测的基础,两种格式可互转
2. **锚框**: 预定义候选框,减少搜索空间
3. **多尺度**: 不同层检测不同大小目标
4. **感受野**: 深层感受野大,适合大目标
5. **批处理**: 填充统一shape,class=-1标记填充

## 练习

1. **边界框转换**: 给定任意边界框,验证两种格式的相互转换。

2. **锚框数量**: 计算不同配置下的锚框总数:
   - 图像: 800×600
   - 尺度: [0.8, 0.6, 0.4, 0.2]
   - 宽高比: [1, 1.5, 2, 0.5]

3. **尺度设计**: 为什么大特征图用小锚框?从感受野角度解释。

4. **数据增广**: 对目标检测数据集应用增广时,如何同步变换边界框?

5. **实例分析**: 在实际图像上手动标注边界框,理解标注难度。

6. **可视化**: 实现在一张图上同时显示多尺度锚框。

7. **自定义数据集**: 创建自己的目标检测数据集,实现Dataset类。

8. **IoU计算**: 实现边界框的交并比(IoU)计算函数。